In [ ]:
import requests
import time
import os
from datetime import datetime, timedelta

timeout = 30000000

BASE_URL = "http://10.252.38.241:5001"

def generate_pdf(folder_name, lightweight=False, custom_plot_config=None):
    """
    Generate PDF report for a folder.
    
    Args:
        folder_name (str): Name of the folder to analyze (e.g., '20250110' - just folder name, not path)
        lightweight (bool): If True, uses lightweight config that disables expensive plots
        custom_plot_config (dict): Optional custom plot configuration to override specific plots.
    
    Returns:
        bool: True if successful, False otherwise
    """
    try:
        # Build analyze request payload
        analyze_payload = {"folder": folder_name}
        
        # Add lightweight mode if requested
        if lightweight:
            analyze_payload["lightweight"] = True
            print(f"Using LIGHTWEIGHT mode for {folder_name}")
        
        # Add custom plot config if provided
        if custom_plot_config:
            analyze_payload["plot_config"] = custom_plot_config
            print(f"Using custom plot config: {custom_plot_config}")
        
        analyze_response = requests.post(
            BASE_URL+"/api/analyze",
            json=analyze_payload,
            timeout=timeout
        )

        if analyze_response.status_code != 200:
            print(f"Analysis failed for {folder_name}. Status code: {analyze_response.status_code}")
            try:
                error_detail = analyze_response.json()
                print(f"Error details: {error_detail}")
            except:
                print(f"Response text: {analyze_response.text}")
            return False

        pdf_response = requests.get(BASE_URL+"/api/export_pdf", timeout=timeout)

        if pdf_response.status_code != 200:
            print(f"Failed to generate PDF for {folder_name}. Status code: {pdf_response.status_code}")
            try:
                error_detail = pdf_response.json()
                print(f"Error details: {error_detail}")
            except:
                print(f"Response text: {pdf_response.text}")
            return False

        pdf_path = f"{folder_name}.pdf"
        with open(pdf_path, 'wb') as pdf_file:
            pdf_file.write(pdf_response.content)
        print(f"PDF saved: {pdf_path}")

        stats_filename = f"{folder_name}_stats.json"
        json_response = requests.get(
            BASE_URL+"/api/export_stats_json",
            params={"filename": stats_filename},
            timeout=timeout
        )

        if json_response.status_code != 200:
            print(f"Failed to generate statistics JSON for {folder_name}. Status code: {json_response.status_code}")
            return False

        with open(stats_filename, 'wb') as json_file:
            json_file.write(json_response.content)
        print(f"Statistics JSON saved: {stats_filename}")

        return True

    except Exception as e:
        print(f"Error generating reports for {folder_name}: {e}")
        return False


def set_plot_config(lightweight=False, custom_config=None):
    """
    Set the global plot configuration on the server.
    """
    if lightweight:
        response = requests.post(BASE_URL+"/api/plot_config", json={"lightweight": True})
    elif custom_config:
        response = requests.post(BASE_URL+"/api/plot_config", json={"config": custom_config})
    else:
        response = requests.post(BASE_URL+"/api/plot_config", json={"full": True})
    
    return response.json()


def get_plot_config():
    """Get current plot configuration from server."""
    response = requests.get(BASE_URL+"/api/plot_config")
    return response.json()


def list_folders():
    """List available folders from the server."""
    response = requests.get(BASE_URL+"/api/folders", timeout=timeout)
    if response.status_code == 200:
        data = response.json()
        print(f"Available folders: {data.get('folders', [])}")
        return data.get('folders', [])
    else:
        print(f"Failed to get folders. Status: {response.status_code}")
        try:
            print(f"Error: {response.json()}")
        except:
            print(f"Response: {response.text}")
        return []

In [ ]:
# First, list available folders to find a valid one
folders = list_folders()

# Then use an actual folder name (NOT 'data/YYYYMMDD')
# The folder name should be just the folder, e.g., '20250204'
if folders:
    directory = folders[0]  # Use the first available folder
    print(f"Generating PDF for: {directory}")
    success = generate_pdf(directory)
else:
    print("No folders available on the server")

In [ ]:
# Check current server plot configuration
config = get_plot_config()
print("Current plot config:", config)

# Switch server to lightweight mode for all subsequent requests
# set_plot_config(lightweight=True)

# Switch back to full mode
# set_plot_config(lightweight=False)